# 循环神经网络（下）双向 LSTM 与变长序列

本节做两件事：

1. **变长序列** 用 `pad_sequence` + `pack_padded_sequence` 处理，避免 LSTM 在 `<pad>` 上做无意义计算。
2. **双向 LSTM**（`bidirectional=True`）在一个合成的"情感分类"任务上展示比单向 LSTM 更强的判别力。

## 1. 合成情感分类任务（信号集中在序列开头）

- 词表 25 个 token：编号 0 是 `<pad>`，1-3 是 "positive" 词（如 `good/great/love`），4-6 是 "negative" 词（如 `bad/awful/hate`），7-24 是中性 filler 词。
- 每条样本：长度 35-45 的中性序列，**前 3 个位置** 全部替换成同极性的情感词（决定标签）。
- 这样设计让"信号"出现在序列**开头**，距离最终的隐状态 35+ 步——这正是单向 LSTM 的长程依赖弱点。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

VOCAB = 25; PAD = 0; POS = [1, 2, 3]; NEG = [4, 5, 6]; FILL = list(range(7, VOCAB))

def make_sample(g):
    L = torch.randint(35, 46, (1,), generator=g).item()
    label = torch.randint(0, 2, (1,), generator=g).item()      # 0 or 1
    tokens = [FILL[torch.randint(0, len(FILL), (1,), generator=g).item()] for _ in range(L)]
    sent_pool = POS if label == 1 else NEG
    for i in range(3):                                          # 前 3 个位置放情感词
        tokens[i] = sent_pool[torch.randint(0, 3, (1,), generator=g).item()]
    return torch.tensor(tokens, dtype=torch.long), label

class SentimentDS(Dataset):
    def __init__(self, n, seed):
        g = torch.Generator().manual_seed(seed)
        self.data = [make_sample(g) for _ in range(n)]
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

ds = SentimentDS(5, seed=0)
for x, y in ds:
    print(f'len={len(x):2d}  label={y}  tokens={x.tolist()}')

## 2. `collate_fn` 处理变长 + 排序

DataLoader 把不同长度的样本组成一个 batch 时，要：

1. **pad** 到这个 batch 的最大长度（用 `pad_sequence`）
2. **记下原始长度**（`pack_padded_sequence` 需要）

PyTorch 1.x 时代 `pack_padded_sequence` 要求按长度降序排好；当前版本传 `enforce_sorted=False` 就自动处理。

In [ ]:
def collate(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(seqs, batch_first=True, padding_value=PAD)
    return padded, lengths, torch.tensor(labels)

train_loader = DataLoader(SentimentDS(2000, seed=0), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader   = DataLoader(SentimentDS(500,  seed=1), batch_size=64, collate_fn=collate)

padded, lengths, labels = next(iter(train_loader))
print(f'padded.shape = {tuple(padded.shape)}')
print(f'lengths      = {lengths[:8].tolist()} ...')
print(f'labels       = {labels[:8].tolist()} ...')

## 3. 模型：单向 vs 双向 LSTM

PyTorch `nn.LSTM(bidirectional=True)` 输出维度是 `2 * hidden`（前向 + 后向拼接）。我们取 `h_n` 在最后一层的 forward + backward 隐状态拼接做分类特征。

In [ ]:
class SentClassifier(nn.Module):
    def __init__(self, vocab=VOCAB, embed=32, hidden=64, bidirectional=False):
        super().__init__()
        self.emb = nn.Embedding(vocab, embed, padding_idx=PAD)
        self.lstm = nn.LSTM(embed, hidden, batch_first=True, bidirectional=bidirectional)
        n_dir = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden * n_dir, 2)
        self.bidirectional = bidirectional

    def forward(self, padded, lengths):
        e = self.emb(padded)
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        # h: [num_layers * n_dir, B, hidden]，最后一层的方向排在末尾
        # 用负索引让代码对 num_layers > 1 也成立
        if self.bidirectional:
            feat = torch.cat([h[-2], h[-1]], dim=1)      # 最后一层 forward + backward
        else:
            feat = h[-1]                                  # 最后一层
        return self.fc(feat)

print('uni  params :', sum(p.numel() for p in SentClassifier(bidirectional=False).parameters()))
print('bi   params :', sum(p.numel() for p in SentClassifier(bidirectional=True).parameters()))

## 4. 训练比较

In [ ]:
def train(model, epochs=8, lr=3e-3):
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    hist = []
    for _ in range(epochs):
        model.train()
        for x, lens, y in train_loader:
            opt.zero_grad()
            loss_fn(model(x, lens), y).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, lens, y in dev_loader:
                correct += (model(x, lens).argmax(1) == y).sum().item()
                total += y.size(0)
        hist.append(correct / total)
    return hist

torch.manual_seed(0)
uni_hist = train(SentClassifier(bidirectional=False))
torch.manual_seed(0)
bi_hist  = train(SentClassifier(bidirectional=True))

for ep, (u, b) in enumerate(zip(uni_hist, bi_hist), 1):
    print(f'epoch {ep}: uni={u:.4f}  bi={b:.4f}')

In [ ]:
plt.plot(uni_hist, '-o', label='unidirectional')
plt.plot(bi_hist,  '-s', label='bidirectional')
plt.xlabel('epoch'); plt.ylabel('dev accuracy'); plt.ylim(0.4, 1.05); plt.grid(alpha=0.3); plt.legend(); plt.show()

**观察**：

- **单向 LSTM 卡在 ~50%（随机水平）**：信号在开头 3 位，序列长 35-45——前向流到达最终隐状态时，开头的信息基本已经被覆盖。
- **双向 LSTM ≈ 100%**：后向流最后看到的恰好是开头的情感词，毫无衰减压力。
- 这是双向架构的典型受益场景。实际 NLP 里 NER / POS tagging / 阅读理解等任务也类似——每个位置的判断都依赖上下文两个方向的信息。

## 5. 关于 `pack_padded_sequence` 的几个易踩点

- `lengths` 参数必须在 **CPU** 上（即使数据在 GPU）。
- 设 `enforce_sorted=False`，否则要先按长度降序排再恢复——很麻烦。
- 解包：`out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)` 拿回 `[B, L, hidden*n_dir]`。
- `padding_idx=PAD` 让 `nn.Embedding` 对 PAD token 的梯度归零，参数不会被 PAD 拖坏。

## 小结

- **变长序列**：`pad_sequence` + `pack_padded_sequence` 是标准做法；LSTM 跳过 PAD 位置的计算。
- **双向 LSTM**：`bidirectional=True`，输出维度 `hidden * 2`；要从开头和末尾都提取信息的任务上受益明显。
- 实际文本分类（IMDB / AG_News 等）流程基本相同：tokenize → vocab + pad → embedding → bi-LSTM → linear → CE loss。
- 下一步（chap7）：网络优化与正则化——更系统地讲 lr、optimizer、weight decay、dropout 等。